# Knowledge Graph Structure

Load and visualize the built knowledge graph for THINGS-EEG2 dataset.

In [ ]:
import os
import sys
from pathlib import Path

import torch
from omegaconf import OmegaConf
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.tree import Tree

console = Console()

In [ ]:
def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "src").exists():
            return candidate
    return start


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("PROJECT_ROOT", str(ROOT))
paths = OmegaConf.load(ROOT / "configs" / "paths" / "default.yaml")
build_kg = OmegaConf.load(ROOT / "configs" / "build_kg" / "thingseeg2.yaml")

# Merge configs to resolve interpolations
config = OmegaConf.merge({"paths": paths}, {"build_kg": build_kg})

k = config.build_kg.k
partition = config.build_kg.partition
use_open_vocab = config.build_kg.get("use_open_vocab", False)
output_format = config.build_kg.get("output_format", "pt")
save_dir = Path(str(config.build_kg.save_dir)).expanduser()

kg_path = save_dir / f"thingseeg2_kg_{partition}_k{k}.pt"

console.print(Panel.fit(str(ROOT), title="Project Root", border_style="cyan"))

config_table = Table(title="Knowledge Graph Config", show_lines=True)
config_table.add_column("Field", style="bold cyan")
config_table.add_column("Value", overflow="fold")
config_table.add_row("k (neighbors per concept)", str(k))
config_table.add_row("partition", partition)
config_table.add_row("use_open_vocab", str(use_open_vocab))
config_table.add_row("output_format", output_format)
config_table.add_row("save_dir", str(save_dir))
config_table.add_row("kg_path", str(kg_path))
config_table.add_row("exists", "yes" if kg_path.exists() else "no")
if kg_path.exists():
    config_table.add_row("size", f"{kg_path.stat().st_size:,} bytes")
console.print(config_table)

In [ ]:
def tensor_summary(value):
    summary = {
        "shape": tuple(value.shape),
        "dtype": str(value.dtype),
        "numel": f"{value.numel():,}",
    }
    if value.numel() > 0 and not value.dtype.is_floating_point:
        summary.update({"min": value.min().item(), "max": value.max().item()})
    return summary


def add_value_to_tree(tree, name, value, max_items=5):
    if torch.is_tensor(value):
        branch = tree.add(f"[bold green]{name}[/]: Tensor")
        for key, item in tensor_summary(value).items():
            branch.add(f"[cyan]{key}[/]: {item}")
    elif isinstance(value, dict):
        branch = tree.add(f"[bold blue]{name}[/]: dict (keys={len(value)})")
        for i, (key, item) in enumerate(list(value.items())[:max_items]):
            add_value_to_tree(branch, str(key), item, max_items=max_items)
        if len(value) > max_items:
            branch.add(f"... {len(value) - max_items} more")
    elif isinstance(value, (list, tuple)):
        branch = tree.add(f"[bold magenta]{name}[/]: {type(value).__name__} (len={len(value)})")
        for idx, item in enumerate(value[:max_items]):
            add_value_to_tree(branch, f"[{idx}]", item, max_items=max_items)
        if len(value) > max_items:
            branch.add(f"... {len(value) - max_items} more")
    else:
        tree.add(f"[bold]{name}[/]: {type(value).__name__} = {value!r}")

In [ ]:
console.rule("[bold cyan]Loading Knowledge Graph")
kg_data = torch.load(kg_path, map_location="cpu", weights_only=False)

tree = Tree("[bold]knowledge_graph[/]")
add_value_to_tree(tree, "root", kg_data, max_items=4)
console.print(tree)

In [ ]:
stats_table = Table(title="Knowledge Graph Statistics", show_lines=True)
stats_table.add_column("Metric", style="bold cyan")
stats_table.add_column("Value", justify="right")

if "concept_neighbors" in kg_data:
    stats_table.add_row("Number of concepts", str(kg_data["concept_neighbors"].shape[0]))
    stats_table.add_row("Neighbors per concept", str(kg_data["concept_neighbors"].shape[1]))
if "image_to_concept" in kg_data:
    stats_table.add_row("Number of images", str(len(kg_data["image_to_concept"])))
if "concept_to_images" in kg_data:
    img_counts = [len(imgs) for imgs in kg_data["concept_to_images"].values()]
    stats_table.add_row("Images per concept (mean)", f"{sum(img_counts) / len(img_counts):.1f}")
    stats_table.add_row("Images per concept (min)", str(min(img_counts)))
    stats_table.add_row("Images per concept (max)", str(max(img_counts)))

console.print(stats_table)

In [ ]:
if "concepts" in kg_data:
    # Open vocabulary mode
    sample_table = Table(
        title="Sample Concept Neighbors (first 5 concepts, 5 neighbors each)", show_lines=True
    )
    sample_table.add_column("Concept ID", style="bold cyan")
    sample_table.add_column("Name", style="white")
    sample_table.add_column("Neighbor Concepts", overflow="fold")

    for concept_id in range(min(5, kg_data["concept_neighbors"].shape[0])):
        name = kg_data["concepts"][concept_id]["name"]
        neighbor_ids = kg_data["concept_neighbors"][concept_id][:5].tolist()
        neighbor_names = [
            kg_data["concepts"][nid]["name"] for nid in neighbor_ids if nid in kg_data["concepts"]
        ]
        sample_table.add_row(str(concept_id), name, str(neighbor_names))

    console.print(sample_table)
elif "concept_to_images" in kg_data:
    # Dataset-only mode
    sample_table = Table(
        title="Sample Concept Neighbors (first 5 concepts, 5 neighbors each)", show_lines=True
    )
    sample_table.add_column("Concept ID", style="bold cyan")
    sample_table.add_column("# Images", justify="right")
    sample_table.add_column("Neighbor Concepts", overflow="fold")

    for concept_id in range(min(5, kg_data["concept_neighbors"].shape[0])):
        n_images = len(kg_data["concept_to_images"][concept_id])
        neighbors = kg_data["concept_neighbors"][concept_id][:5].tolist()
        sample_table.add_row(str(concept_id), str(n_images), str(neighbors))

    console.print(sample_table)